# FGAI (ICT4403) Assignment 3 - EN→HI MT on **Kaggle**
Before running: Settings → Accelerator **GPU T4 x2** · Internet **On** · Add-ons → Secrets → `HF_TOKEN` attached · Input → your `en-hi-mt-comparison` dataset added.
First run with `TINY_RUN = True` (minutes) to check everything, then do the real run with **Save Version → Save & Run All (Commit)** (fresh container, keeps running with the tab closed).

## 1. Settings

In [ ]:
TINY_RUN = False          # True = tiny pipeline check (minutes).  False = the real experiment.
CLEAN_CHECKPOINTS = True  # delete model weights at the end so the notebook output stays small

# "Light" preset = smaller training subsets (Llama-3.2-1B is already the smallest Llama 3.x).
# Set LIGHT = False for the full-size experiment (Gemma-3 1B, more training data).
LIGHT = True
LIGHT_SETTINGS = {
    # "GEMMA_ID": "google/gemma-3-270m",   # optional even smaller Gemma (accept its licence first)
    "N_TRAIN_SCRATCH": "50000", "SCRATCH_EPOCHS": "10",
    "N_TRAIN_MT5": "20000",     "MT5_EPOCHS": "2",
    "N_TRAIN_LLM": "5000",
    "N_TEMP_TEST": "100",
}
OVERRIDES = dict(LIGHT_SETTINGS) if LIGHT else {}
# Add your own overrides here if a stage is still too slow, e.g.  OVERRIDES["N_TRAIN_MT5"] = "10000"

TINY = dict(POOL_SIZE="2000", N_TEST="50", N_TEMP_TEST="20", N_TRAIN_SCRATCH="2000", N_TRAIN_MT5="1000",
            N_TRAIN_LLM="500", SCRATCH_EPOCHS="2", MT5_EPOCHS="1", SCRATCH_VOCAB="4000")

## 2. Copy the code into /kaggle/working and check the GPU

In [ ]:
import glob, os, shutil, subprocess, sys, torch

hits = glob.glob("/kaggle/input/**/src/config.py", recursive=True)
assert hits, "Repo not found: add your en-hi-mt-comparison dataset under Input (right sidebar -> Add Input)."
SRC_ROOT = os.path.dirname(os.path.dirname(hits[0]))
REPO = "/kaggle/working/en-hi-mt-comparison"
if os.path.exists(REPO):
    shutil.rmtree(REPO)                      # always start from a clean copy
shutil.copytree(SRC_ROOT, REPO)
os.chdir(REPO)
print("repo copied from", SRC_ROOT, "->", REPO)

assert torch.cuda.is_available(), "No GPU! Settings -> Accelerator -> GPU T4 x2, then restart the session."
print("GPU:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())

## 3. Install pinned dependencies

In [ ]:
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "pip install failed"
# peft + the old torchao that Kaggle ships are incompatible (LoRA stages crash) - torchao is not needed, remove it
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], capture_output=True)
import importlib.metadata as md_
for p in ["torch", "transformers", "peft", "datasets", "sacrebleu", "sentencepiece"]:
    print(p, md_.version(p))

## 4. Hugging Face token + access check (fails fast instead of after hours)

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

from huggingface_hub import hf_hub_download
hf_hub_download("cfilt/iitb-english-hindi", "README.md", repo_type="dataset"); print("OK dataset reachable (internet is on)")
for repo_id in [OVERRIDES.get("LLAMA_ID", "meta-llama/Llama-3.2-1B-Instruct"), OVERRIDES.get("GEMMA_ID", "google/gemma-3-1b-it")]:
    try:
        hf_hub_download(repo_id, "config.json"); print("OK access:", repo_id)
    except Exception as e:
        raise RuntimeError(f"No access to {repo_id}. Open its page on huggingface.co, accept the licence with the SAME account "
                           f"as your token, wait for approval, then rerun. ({type(e).__name__})")

## 5. Helper

In [ ]:
os.environ.update(OVERRIDES)
if TINY_RUN:
    os.environ.update(TINY)          # tiny sizes win over OVERRIDES (but GEMMA_ID stays)
FAILED = []
def run(cmd, required=True):
    """Run a shell command and STREAM its output into the cell (subprocess output is otherwise invisible in notebooks)."""
    print("\n>>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        FAILED.append(cmd)
        if required:
            raise RuntimeError(f"FAILED: {cmd}")
        print(f"FAILED: {cmd} (continuing)")

## 6. Data

In [ ]:
run('python -m src.data')

## 7. Model 1 - Basic Transformer (from scratch)

In [ ]:
run('python -m src.train_transformer')

## 8. Model 2 - mT5 fine-tuning

In [ ]:
run('python -m src.finetune_mt5')

## 9. Model 3 - LLaMA (LoRA)

In [ ]:
run('python -m src.llm_mt --family llama --mode lora')

## 10. Model 4 - Gemma (LoRA)

In [ ]:
run('python -m src.llm_mt --family gemma --mode lora', required=False)

## 11. Prompting baselines (optional)

In [ ]:
for fam in ["llama", "gemma"]:
    for mode in ["zeroshot", "fewshot"]:
        run(f"python -m src.llm_mt --family {fam} --mode {mode} --skip_temp", required=False)

## 12. Tokenizer stats, tables, plots, README

In [ ]:
run('python -m src.tokenizer_stats', required=False)
run('python -m src.compare')
print("\nFailed commands:", FAILED or "none")

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("results/results_table.md").read()))
for f in ["quality_comparison", "compute_comparison", "temperature_analysis", "training_curves"]:
    if os.path.exists(f"results/{f}.png"):
        display(Image(f"results/{f}.png"))

## 13. Bundle the results for download

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/results_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk("results"):
        for f in files:
            z.write(os.path.join(root, f))
    z.write("README.md")
if CLEAN_CHECKPOINTS:
    shutil.rmtree("checkpoints", ignore_errors=True)
print("Done. Download results_bundle.zip from the notebook's Output tab.")